<a href="https://colab.research.google.com/github/winter-pro/Statistical-Learning-e23091/blob/main/Assignment_7d_Structural_Health_Monitoring.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Physical Likelihood FormulationAt time step $k$, a physical sensor records a continuous measurement $Y_k = y_k$ (such as dynamic modal frequency, strain measurement, or vibrational response).The measurement model is governed by a known structural forward response function $g(\theta)$ subject to additive Gaussian sensor noise $\epsilon_k \sim \mathscr{N}(0, \sigma^2)$:$$Y_k = g(\theta) + \epsilon_k$$Conditional on the underlying structural integrity parameter $\Theta = \theta$, the likelihood contribution of a single observation $y_k$ at step $k$ is given by the Gaussian probability density function:$$L(y_k \mid \theta) = f_{Y_k \mid \Theta}(y_k \mid \theta) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left( -\frac{\left(y_k - g(\theta)\right)^2}{2\sigma^2} \right)$$2. Sequential Likelihood and Joint HistoryLet $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)^T$ represent the vector of sequential sensor observations gathered up to step $k$.Assuming that sensor noise terms across consecutive time steps are conditionally independent given $\Theta = \theta$, the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$ is:$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^k L(y_i \mid \theta) = \frac{1}{(2\pi\sigma^2)^{k/2}} \exp\left( -\frac{1}{2\sigma^2} \sum_{i=1}^k \left(y_i - g(\theta)\right)^2 \right)$$3. Mathematical Formulation of Bounded Recursive UpdatesBefore observing measurements, the platform initializes a prior density function $f_{\Theta}^{(0)}(\theta)$ over the physically bounded interval $[\theta_{\min}, \theta_{\max}]$ (e.g., a uniform distribution $U(\theta_{\min}, \theta_{\max})$ indicating initial uninformative uncertainty).Under a sequential Bayesian updating framework, the posterior distribution at step $k-1$ serves as the prior distribution for step $k$. The running posterior density function $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$ is recursively updated via Bayes' Theorem:$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) = \frac{L(y_k \mid \theta) \, f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})}{\int_{\theta_{\min}}^{\theta_{\max}} L(y_k \mid s) \, f_{\Theta \mid \mathbf{Y}^{(k-1)}}(s \mid \mathbf{y}^{(k-1)}) \, ds}$$Definition of Key Components:$f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$: The prior density at step $k$ inherited directly from the previous state $k-1$.$L(y_k \mid \theta)$: The likelihood contribution of the incoming real-time sensor reading $y_k$.Denominator (Normalizing Constant $Z_k$): Integrates the product of likelihood and prior across the bounded domain $[\theta_{\min}, \theta_{\max}]$ to ensure the total area under the density curve equals $1$.4. Running Point EstimatorsFrom the running posterior distribution $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$, two primary estimators track structural health at step $k$:a. Running Posterior Mean ($\widehat{\theta}_{\text{Bayes}}^{(k)}$)Under a squared-error loss function, the optimal point estimate is the expected value of the current bounded posterior distribution:$$\widehat{\theta}_{\text{Bayes}}^{(k)} = \mathbb{E}\left[\Theta \mid \mathbf{Y}^{(k)} = \mathbf{y}^{(k)}\right] = \int_{\theta_{\min}}^{\theta_{\max}} \theta \, f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \, d\theta$$b. Running Maximum A Posteriori ($\widehat{\theta}_{\text{MAP}}^{(k)}$)The most probable structural state corresponds to the peak (mode) of the current posterior density over the bounded domain:$$\widehat{\theta}_{\text{MAP}}^{(k)} = \arg\max_{\theta \in [\theta_{\min}, \theta_{\max}]} f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$$5. Numerical Implementation via Bounded Grid DiscretizationSince non-linear structural response functions $g(\theta)$ generally lack closed-form analytical conjugate solutions, the system maintains the posterior on a fine discrete grid across the bounded physical domain $[\theta_{\min}, \theta_{\max}]$.Algorithmic Procedure:a. Grid Setup: Define $M$ equally spaced grid points over $[\theta_{\min}, \theta_{\max}]$:$$\theta_m = \theta_{\min} + (m-1)\Delta\theta, \quad \text{where } \Delta\theta = \frac{\theta_{\max} - \theta_{\min}}{M-1}, \quad m = 1, 2, \dots, M$$b. Prior Initialization: Evaluate initial prior values across the grid array $\mathbf{P}_0 = [P_0(\theta_1), \dots, P_0(\theta_M)]$ and normalize using numerical integration (e.g., composite trapezoidal rule):$$Z_0 = \text{trapezoid}(\mathbf{P}_0, \boldsymbol{\theta}), \quad \mathbf{P}_0 \leftarrow \frac{\mathbf{P}_0}{Z_0}$$c. Sequential Updating & Normalization (at step $k$):Compute unnormalized posterior array: $\tilde{P}_k(\theta_m) = P_{k-1}(\theta_m) \times L(y_k \mid \theta_m)$Compute normalizing factor: $Z_k = \sum_{m=1}^{M-1} \frac{\tilde{P}_k(\theta_m) + \tilde{P}_k(\theta_{m+1})}{2} \Delta\theta$Normalize: $P_k(\theta_m) = \frac{\tilde{P}_k(\theta_m)}{Z_k}$d. Point Estimation Evaluation:Bayes Estimate: $\widehat{\theta}_{\text{Bayes}}^{(k)} \approx \text{trapezoid}(\boldsymbol{\theta} \odot \mathbf{P}_k, \boldsymbol{\theta})$MAP Estimate: $\widehat{\theta}_{\text{MAP}}^{(k)} = \theta_{m^*}, \quad \text{where } m^* = \arg\max_{m} P_k(\theta_m)$6. Dynamic Mechanics & Convergence InterpretationVariance Reduction: As $k$ increases, repeated noisy sensor measurements progressively attenuate likelihood ambiguity, tightening the posterior distribution variance around the true structural state $\theta_{\text{true}}$.Physical Boundary Enforcement: The bounded grid framework strictly guarantees that probability mass outside $[\theta_{\min}, \theta_{\max}]$ is zero, avoiding physically unfeasible state estimates (such as negative stiffness or over-100% health).Sensor Noise vs. Tracking Sensitivity: Lower measurement noise $\sigma$ narrows the single-step likelihood curve $L(y_k \mid \theta)$, allowing rapid posterior convergence with fewer sensor readings.

In [7]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# Set random seed for reproducibility
np.random.seed(24)

# =====================================================================
# CONFIGURATION & PARAMETERS
# =====================================================================
theta_true = 0.68       # True remaining stiffness efficiency of the beam (68%)
K_nominal = 50.0        # Nominal baseline stiffness of the pristine structure (kN/mm)
sigma = 0.15            # Sensor noise standard deviation (log-space)
n_sensor_readings = 15  # Timeline steps

# 1. Define a fine grid over the physical boundary [0.01, 1.0]
theta_grid = np.linspace(0.01, 1.0, 500)

# 2. Initialize Prior: Bounded Beta distribution reflecting an initially healthy beam
# centered heavily near 0.95-1.0
current_posterior = stats.beta.pdf(theta_grid, a=8, b=1.5)
# Normalize initial prior
current_posterior /= np.trapezoid(current_posterior, theta_grid)

# Steps milestone tracking for plotting curves
milestones = [0, 1, 2, 5, 10, 15]

# Create Figure
fig = go.Figure()

# Plot Initial Prior State
fig.add_trace(go.Scatter(
    x=theta_grid, y=current_posterior, mode='lines',
    name='Prior State: Structural Health Assumed Healthy',
    line=dict(dash='dash', width=2.5, color='gray')
))

# =====================================================================
# SEQUENTIAL BAYESIAN MONITORING LOOP
# =====================================================================
for k in range(1, n_sensor_readings + 1):
    # Simulate a noisy structural sensor reading from log-normal physics
    noise = np.random.normal(0, sigma)
    y_k = (theta_true * K_nominal) * np.exp(noise)

    # Calculate Log-Normal Likelihood curve across the structural theta grid
    # Expected value for any grid point is: grid_point * K_nominal
    expected_K = theta_grid * K_nominal
    likelihood = stats.lognorm.pdf(y_k, s=sigma, scale=expected_K)

    # Running Update: Posterior Proportional to Prior * Likelihood
    current_posterior = current_posterior * likelihood

    # Numerical Normalization via Trapezoidal rule
    integral = np.trapezoid(current_posterior, theta_grid)
    current_posterior /= integral

    # Capture structural health density profile at milestones
    if k in milestones:
        fig.add_trace(go.Scatter(
            x=theta_grid, y=current_posterior, mode='lines',
            name=f"Step {k}: Post-Sensor Reading (Observed K={y_k:.2f})",
            line=dict(width=2)
        ))

# =====================================================================
# VISUALIZE STRUCTURAL DEGRADATION TRACKING
# =====================================================================
fig.add_vline(
    x=theta_true, line_dash="dot", line_color="red", line_width=2.5,
    annotation_text=f"True Structural Degradation State ({theta_true})",
    annotation_position="top left"
)

fig.update_layout(
    title={
        'text': "Structural Health Monitoring: Bounded Bayesian Parameter Updating",
        'y': 0.95, 'x': 0.5, 'xanchor': 'center', 'yanchor': 'top'
    },
    xaxis_title="Remaining Structural Stiffness Efficiency Factor (θ)",
    yaxis_title="Probability Density (Confidence level of damage)",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(
        yanchor="top", y=0.95, xanchor="left", x=0.02,
        bgcolor="rgba(255,255,255,0.7)"
    )
)

fig.show()